# STAR testing - Hamilton STAR / STARLet

Validation notebook for the v1 STAR (`pylabrobot.hamilton.star`).

It drives a `STARDevice` - the instrument as a resource, with its deck as its child. The driver
stays reachable underneath as `star.driver`, and machine-level reads that the device does not
proxy are sent through it.

**`setup()` moves the machine.** Watch the log: it reports each phase at `DEBUG` and the machine
it found at `INFO`. It runs in three steps:

1. **discover** - read-only. Machine configuration, arm geometry, channel count, and every
   channel's firmware, width and installed hardware.
2. **initialize** - `C0 VI` on a machine that is not initialized, which homes every drive; or
   `C0 ZA` alone on one that is, to raise the channels to Z safety.
3. **capability bring-up** - the channels eject whatever is mounted on them, including grippers.

If you want to connect and look without anything moving, run `discover()` on its own; the cell
below shows how.

Set `protocol_mode` to `"simulation"` to run every cell against a simulated STAR - no hardware,
no USB, and the same code paths as the real driver.

## 1- Run identity

In [1]:
# --- Run identity ---
protocol_mode = "execution"  # simulation OR execution
user_name = "star_user"
run_identifier = "star_v1_validation"

# --- Which instrument ---
# One of the factories in pylabrobot.hamilton.star.device. It fixes the machine's footprint and
# where its deck sits inside it, and builds the matching deck.
instrument = "STAR"  # STAR OR STARLet OR STAR_with_extension_housing

# --- Device selection (only needed with more than one Hamilton on USB) ---
device_address = None  # USB address, e.g. 3
serial_number = None  # USB serial, e.g. "1234567"

# --- Motion ---
# The X-arm move near the end of this notebook only runs when this is True.
allow_x_arm_move = False

# --- 96-head ---
# Where the 96-head ejects when it is initialized: head channel A1, in deck mm. Initializing it
# throws off whatever is mounted, so this has to be somewhere tips may be dropped, which depends
# on where the waste sits on this deck - hence no default. Setup initializes the head when this is
# set, and reports that it cannot when it is None. This machine was last sent (-263.8, 108.3,
# 200.0), read off the `C0 EI` command in an earlier run.
head96_initialize_position = (-263.8, 108.3, 200.0)

## 2- Imports

In [2]:
from pylabrobot.hamilton.star.device import STAR, STAR_with_extension_housing, STARLet
from pylabrobot.hamilton.star.driver.features.head96 import Head96
from pylabrobot.hamilton.star.driver.master import STARDriver

## 3- Logging

Uses PyLabRobot's own `setup_logger`, exactly as every other PLR run does: a single
date-stamped file per day, appended to across runs. Both the file and the notebook are at
`IO` level, so every byte sent to and received from the machine is visible and recorded.

Re-running this cell is safe: `setup_logger` replaces the file handler and `verbose`
replaces the console handler, rather than stacking a second one of each.

In [3]:
import logging

import pylabrobot
from pylabrobot.io import LOG_LEVEL_IO

log_dir = f"_logs/{protocol_mode}"

# PLR's own logger setup: one date-stamped file per day, appended to across runs. Re-running this
# cell replaces the file handler rather than stacking a second one, so lines are never duplicated.
pylabrobot.setup_logger(log_dir, level=LOG_LEVEL_IO)

# Console at IO level too: every byte sent and received appears in the notebook.
pylabrobot.verbose(True, level=LOG_LEVEL_IO)

print(f"appending to {log_dir}/pylabrobot-<YYYYMMDD>.log")
logging.getLogger("pylabrobot").info("--- %s (%s) ---", run_identifier, protocol_mode)

2026-08-19 14:41:24,544 - pylabrobot - INFO - --- star_v1_validation (execution) ---


appending to _logs/execution/pylabrobot-<YYYYMMDD>.log


## 4- Connect and bring the machine up

In simulation this is a `STARSimulationDriver`, which answers as a real instrument does - the
same command assembly, error decoding and response parsing run either way.

Set `head96_initialize_position` above for setup to initialize the 96-head too; without it, setup
brings everything else up and reports that it could not do the head.

To connect **without moving anything**, replace `await star.setup()` with:

```python
await star._open()
star._connected = True
await star.discover()
```

In [4]:
build = {
  "STAR": STAR,
  "STARLet": STARLet,
  "STAR_with_extension_housing": STAR_with_extension_housing,
}[instrument]

# The instrument builds its own deck and hands it to the driver, which models the machine into it.
# A simulated one answers from that model, so it is built here rather than passed in.
if protocol_mode == "execution":
  star = build(driver=STARDriver(device_address=device_address, serial_number=serial_number))
else:
  star = build(simulation=True)

# Setup builds each capability the machine turns out to have, but not over one that is already
# there - so a capability configured here keeps its configuration. In simulation the head is
# already there and answers for itself, so configure that one rather than replacing it.
if head96_initialize_position is not None:
  if star.driver.head96 is None:
    star.driver.head96 = Head96(star.driver)
  star.head96.configuration.initialize_position = head96_initialize_position

await star.setup()

print(star)
# setup logs this summary at INFO; printed here too so it is the first thing you see.
print(star.driver.format_setup_summary())

2026-08-19 14:41:24,567 - pylabrobot.hamilton.star.driver.master - DEBUG - Setting up STAR on USB 0x08af:0x8000 ...
2026-08-19 14:41:24,569 - pylabrobot.io.usb - INFO - Finding USB device...
2026-08-19 14:41:24,594 - pylabrobot.io.usb - INFO - Found USB device.
2026-08-19 14:41:24,600 - pylabrobot.io.usb - INFO - Found endpoints. 
Write:
       ENDPOINT 0x2: Bulk OUT ===============================
       bLength          :    0x7 (7 bytes)
       bDescriptorType  :    0x5 Endpoint
       bEndpointAddress :    0x2 OUT
       bmAttributes     :    0x2 Bulk
       wMaxPacketSize   :   0x40 (64 bytes)
       bInterval        :    0x0 
Read:
       ENDPOINT 0x81: Bulk IN ===============================
       bLength          :    0x7 (7 bytes)
       bDescriptorType  :    0x5 Endpoint
       bEndpointAddress :   0x81 IN
       bmAttributes     :    0x2 Bulk
       wMaxPacketSize   :   0x40 (64 bytes)
       bInterval        :    0x0
2026-08-19 14:41:27,604 - pylabrobot.hamilton.star.drive

Hamilton Microlab STAR(STARDriver, 56-track deck)
[Hamilton STAR] Connected on USB 0x08af:0x8000
  Firmware: master 7.6S 25 2021_11_05 (GRU C0), pipettes 4.0S j 2022-03-16, x_arm 1.4S 2012-04-25, head96 5.0S i 2021-10-22 (H0 XE167), iswap 4.1S 2011-12-19, autoload 3.4S f 2017-01-09
  Configuration: 54 slots
  Autoload: 1D barcode scanner
  Arms: 1
    left: hamilton_legacy_star_dual_rail_arm, 354.0 mm wide, travel 95.0 to 1340.2 mm, workspace -323.2 to 1517.2 mm
      channels: 8 (1000uL) | 96-head: 96 head II | 384-head: none | iSWAP: wide gripper


In [5]:
deck = star.deck
deck

HamiltonSTARDeck(name='deck', location=Coordinate(121.800, 116.000, 078.500), size_x=1545, size_y=653.5, size_z=900, category=deck)

In [6]:
star.driver.deck

HamiltonSTARDeck(name='deck', location=Coordinate(121.800, 116.000, 078.500), size_x=1545, size_y=653.5, size_z=900, category=deck)

## 5- The rest of the configuration

What the setup summary does not already print, plus a check of this machine's firmware stack
against the stacks this driver has been driven with before.

In [7]:
from pylabrobot.hamilton.star.driver.confirmed_firmware_versions import suggest_entry, unconfirmed

c = star.driver.configuration
print(f"wash stations         : 1={c.wash_station_1_installed}  2={c.wash_station_2_installed}")
print(f"tip waste x           : {c.tip_waste_x_position} mm")
print(
  f"iSWAP collision-free  : {c.min_iswap_collision_free_position} to "
  f"{c.max_iswap_collision_free_position} mm"
)
print(f"pip maximal y         : {c.pip_maximal_y_position} mm")
print(f"initialized           : {await star.driver.request_initialization_status()}")

# Has each of this machine's boards been driven on the firmware it reports?
new = unconfirmed(star.driver.firmware)
print()
if not new:
  print(f"firmware: all {len(star.driver.firmware)} capabilities confirmed")
else:
  print(f"firmware: {len(new)} of {len(star.driver.firmware)} capabilities not seen before.")
  print("if this machine works, add them to confirmed_firmware_versions.py:")
  for capability, version in new.items():
    print(suggest_entry(capability, version))

2026-08-19 14:41:30,783 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'C0QWid0063'


wash stations         : 1=False  2=False
tip waste x           : 1340.0 mm
iSWAP collision-free  : 350.0 to 1140.0 mm
pip maximal y         : 606.5 mm


2026-08-19 14:41:30,840 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'C0QWid0063er00/00qw1')


initialized           : True

firmware: all 6 capabilities confirmed


## 6- The X-arms

A STAR always has a left arm and may have a right one. `star.x_arm` is the arm on a machine that
has only one, and refuses on a machine that has two.

In [8]:
for arm in (star.left_x_arm, star.right_x_arm):
  if arm is None:
    print("right: not installed")
    continue
  a = arm.configuration
  print(f"{arm.side:5s}: {a.model}   firmware {a.firmware_version}")
  print(f"       width {a.width} mm, travel {a.x_range} mm, workspace {a.workspace_range} mm")
  print(f"       wrap {a.wrap_size} mm, reference point: {a.reference_point}")
  print(
    f"       modules: pip={a.pip_installed} iswap={a.iswap_installed} "
    f"head96={a.head96_installed} xl={a.xl_channels_installed}"
  )

try:
  print(f"\nstar.x_arm -> {star.x_arm.side}")
except ValueError as e:
  print(f"\nstar.x_arm -> {e}")

left : hamilton_legacy_star_dual_rail_arm   firmware 1.4S 2012-04-25
       width 354.0 mm, travel (95.0, 1340.2) mm, workspace (-323.2, 1517.2) mm
       wrap 595.2 mm, reference point: center
       modules: pip=True iswap=True head96=True xl=False
right: not installed

star.x_arm -> left


## 7- The pipetting channels

`configuration` holds what every channel shares; `configuration.channels` holds one entry per
channel, read off the channel itself during discovery. A machine that reports no channels - a
96-head-only STAR - has no `star.pipettes` at all.

In [9]:
if star.pipettes is None:
  print("no channels installed")
else:
  p = star.pipettes.configuration
  print("shared by every channel:")
  print(f"  y drive   {p.y_drive_mm_per_increment} mm/increment")
  print(f"  z drive   {p.z_drive_mm_per_increment} mm/increment")
  print(f"  dispense  {p.dispensing_drive_uL_per_increment} uL/increment")
  print()
  print(
    f"{'ch':>3}  {'firmware':<20} {'width':>7}  {'channel':<12} {'head':<12} {'stop disc':<10} adc"
  )
  for i, ch in enumerate(p.channels):
    print(
      f"{i:>3}  {str(ch.firmware_version):<20} {str(ch.width):>7}  {str(ch.channel_type):<12} "
      f"{str(ch.head_type):<12} {str(ch.stop_disc_type):<10} {ch.pressure_adc}"
    )

shared by every channel:
  y drive   0.046302083 mm/increment
  z drive   0.01072765 mm/increment
  dispense  0.046876 uL/increment

 ch  firmware               width  channel      head         stop disc  adc
  0  4.0S j 2022-03-16       8.98  ML_STAR      ML_STAR      core_ii    Renesas_X9268
  1  4.0S j 2022-03-16       8.98  ML_STAR      ML_STAR      core_ii    Renesas_X9268
  2  4.0S j 2022-03-16       8.98  ML_STAR      ML_STAR      core_ii    Renesas_X9268
  3  4.0S j 2022-03-16       8.98  ML_STAR      ML_STAR      core_ii    Renesas_X9268
  4  4.0S j 2022-03-16       8.98  ML_STAR      ML_STAR      core_ii    Renesas_X9268
  5  4.0S j 2022-03-16       8.98  ML_STAR      ML_STAR      core_ii    Renesas_X9268
  6  4.0S j 2022-03-16       8.98  ML_STAR      ML_STAR      core_ii    Renesas_X9268
  7  4.0S j 2022-03-16       8.98  ML_STAR      ML_STAR      core_ii    Renesas_X9268


## 8- Sensor read: tip presence

Each channel's sleeve sensor reports whether a tip is mounted. This reads sensors; it does not
move a channel. After a full setup every channel should be empty: the channel initialization
ejects whatever was on them.

In [10]:
presence = await star.driver.request_tip_presence()
for channel, has_tip in enumerate(presence):
  print(f"  channel {channel}: {'tip' if has_tip else '-'}")

2026-08-19 14:41:30,877 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'C0RTid0064'
2026-08-19 14:41:30,921 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'C0RTid0064er00/00rt0 0 0 0 0 0 0 0')


  channel 0: -
  channel 1: -
  channel 2: -
  channel 3: -
  channel 4: -
  channel 5: -
  channel 6: -
  channel 7: -


## 9- The front cover

Two read-only commands, and what they mean is exactly what this check is for.

`C0 RW` reports three inputs, the first of them the cover input. `C0 QC` reports the cover
position. Neither says whether a cover is *fitted*: the master acts only on its non-volatile
configuration, so `main_front_cover_monitoring_installed` is what decides whether the cover is
watched at all, and `star.front_cover` exists only when it is set.

This machine reports it as not installed while the cover and its switch are physically there, so
`QC` is sent raw below rather than through the capability.

**Run this three times and record what changes**: cover shut, cover open, and cover cable
disconnected. If the cover input tracks the position it is a position input; if it holds while
the position changes it is a presence input; if neither moves, the master is not reading the
switch at all - which is what a configuration that says the monitoring is not installed predicts.

That last outcome is the one that decides whether `FrontCover` is worth keeping: a machine that
answers nothing here has no cover to drive, and the capability would only ever be an empty
`request_position` on machines configured differently from this one.


In [11]:
cover_input, second_input, reserve_input = await star.driver.request_cover_input_status()
print(f"inputs        : cover={cover_input}  second={second_input}  reserve={reserve_input}")

c = star.driver.configuration
print(
  f"monitoring    : main={c.main_front_cover_monitoring_installed}"
  f"  additional={c.additional_front_cover_monitoring_installed}"
)
print(f"covers        : left={c.left_cover_installed}  right={c.right_cover_installed}")
print(f"capability    : {star.front_cover}")

# C0 QC - request cover position. Read-only, and sent raw so it answers even on a machine whose
# configuration says the monitoring is not installed.
print(f"position (raw): {await star.driver.send_raw_command('C0QCid9989')}")
if star.front_cover is not None:
  print(f"position      : {await star.front_cover.request_position()}")

2026-08-19 14:41:30,932 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'C0RWid0065'
2026-08-19 14:41:30,952 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'C0RWid0065er00/00rw000')
2026-08-19 14:41:30,957 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'C0QCid9989'


inputs        : cover=False  second=False  reserve=False
monitoring    : main=False  additional=False
covers        : left=False  right=False
capability    : None


2026-08-19 14:41:30,976 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'C0QCid9989er00/00qc1')


position (raw): C0QCid9989er00/00qc1


## 10- Move the X-arm

**This moves the arm and everything mounted on it.** Only run it with the deck clear along the
path, and only after setup has raised the channels to Z safety.

Gated on `allow_x_arm_move`, set at the top of the notebook.

In [12]:
arm = star.x_arm
print(f"travel range: {arm.configuration.x_range} mm")

target = 500.0
if allow_x_arm_move:
  await arm.move_x(target)
  print(f"moved to {target} mm")
else:
  print(f"skipped. set allow_x_arm_move = True to move to {target} mm")

# out-of-range targets are refused before anything reaches the wire
try:
  await arm.move_x(5000.0)
except ValueError as e:
  print("guard:", e)

travel range: (95.0, 1340.2) mm
skipped. set allow_x_arm_move = True to move to 500.0 mm
guard: left X-arm x=5000.0mm is outside its drive travel range [95.0, 1340.2].


In [13]:
deck.get_resource("left_x_arm").get_location_wrt(deck)

Coordinate(x=423.0, y=0.0, z=334.7)

In [14]:
await star.x_arm.request_position(), deck.get_resource("left_x_arm").get_location_wrt(deck)

2026-08-19 14:41:31,013 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid0066'
2026-08-19 14:41:31,022 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0RXid0066rx+06000 +000060000')


(600.0, Coordinate(x=423.0, y=0.0, z=334.7))

In [15]:
(
  await star.driver.send_command(module="C0", command="RX"),
  await star.driver.send_command(module="C0", command="QX"),
)

2026-08-19 14:41:31,031 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'C0RXid0067'
2026-08-19 14:41:31,052 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'C0RXid0067er00/00rx+06000')
2026-08-19 14:41:31,055 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'C0QXid0068'
2026-08-19 14:41:31,076 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'C0QXid0068er00/00rx+00000')


('C0RXid0067er00/00rx+06000', 'C0QXid0068er00/00rx+00000')

In [16]:
# The same gate as the section above: this moves the arm.
allow_x_arm_move = True
if allow_x_arm_move:
  await star.x_arm.move_x(500.0)
  print("moved to 500.0 mm")
else:
  print("skipped. set allow_x_arm_move = True to move")

# What the machine says, and where the model puts the arm's reference point. They should agree.
position = await star.x_arm.request_position()
arm_resource = deck.get_resource("left_x_arm")
seated = arm_resource.get_location_wrt(deck)
print(f"machine: {position} mm")
print(f"model  : {seated.x + arm_resource.get_anchor(x=star.x_arm.reference_anchor).x} mm")

2026-08-19 14:41:31,101 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0XPid0069la05000lr3lw7'
2026-08-19 14:41:32,313 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0XPid0069er00')
2026-08-19 14:41:32,317 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid0070'
2026-08-19 14:41:32,327 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0RXid0070rx+05002 +000050020')
2026-08-19 14:41:32,330 - pylabrobot.hamilton.star.driver.features.x_arm - DEBUG - the left X-arm was sent to 500.0 mm and came to rest at 500.2 mm
2026-08-19 14:41:32,334 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid0071'
2026-08-19 14:41:32,343 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0RXid0071rx+05002 +000050015')


moved to 500.0 mm
machine: 500.2 mm
model  : 500.2 mm


In [17]:
deck.get_resource("left_x_arm").get_size_x() / 2

177.0

## 11- Who closes the loop on an X move?

`X0 XP` asked for 500.0 mm and the arm came to rest at 499.8 - two increments short. The master has
its own absolute move for the same axis, `C0 JX`, and a variant that raises everything to Z safety
first, `C0 KX`. If the master runs a control loop to the target, its move should land closer than
the board's.

**This moves the arm.** Gated on `allow_x_arm_move`. It drives to each target twice, once through
each command, reading back where the arm actually stopped, and reports the residual.


In [18]:
targets = [500.0, 800.0, 1_000.0]

if not allow_x_arm_move:
  print("skipped. set allow_x_arm_move = True to run")
else:
  print(f"{'target':>8} {'X0 XP':>10} {'C0 JX':>10}   residual, mm")
  for target in targets:
    increments = f"{round(target * 10):05}"

    await star.x_arm.move_x(target)  # X0 XP, and it reads back
    by_board = await star.x_arm.request_position()

    # move away first, so the second command has the same distance to cover as the first did not
    await star.x_arm.move_x(target - 50.0)
    await star.driver.send_command(module="C0", command="JX", xs=increments)
    by_master = await star.x_arm.request_position()

    print(f"{target:8.1f} {by_board - target:10.2f} {by_master - target:10.2f}")

  target      X0 XP      C0 JX   residual, mm


2026-08-19 14:41:32,383 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0XPid0072la05000lr3lw7'
2026-08-19 14:41:32,540 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0XPid0072er00')
2026-08-19 14:41:32,545 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid0073'
2026-08-19 14:41:32,554 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0RXid0073rx+05000 +000050001')
2026-08-19 14:41:32,558 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid0074'
2026-08-19 14:41:32,568 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0RXid0074rx+05000 +000050000')
2026-08-19 14:41:32,571 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0XPid0075la04500lr3lw7'
2026-08-19 14:41:33,534 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0XPid0075er00')
2026-08-19 14:41:33,539 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid0076'
2026-08-19 14:41:33,549 - pylabrobot.io.usb - IO - [0x8af:0x8000][][]

   500.0       0.00      -0.30


2026-08-19 14:41:36,165 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0XPid0079er00')
2026-08-19 14:41:36,170 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid0080'
2026-08-19 14:41:36,181 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0RXid0080rx+07998 +000079975')
2026-08-19 14:41:36,184 - pylabrobot.hamilton.star.driver.features.x_arm - DEBUG - the left X-arm was sent to 800.0 mm and came to rest at 799.8 mm
2026-08-19 14:41:36,187 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid0081'
2026-08-19 14:41:36,197 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0RXid0081rx+07998 +000079980')
2026-08-19 14:41:36,201 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0XPid0082la07500lr3lw7'
2026-08-19 14:41:37,162 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0XPid0082er00')
2026-08-19 14:41:37,167 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid0083'
2026-08-19 14:41:37,176 -

   800.0      -0.20      -0.30


2026-08-19 14:41:39,641 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0XPid0086er00')
2026-08-19 14:41:39,646 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid0087'
2026-08-19 14:41:39,655 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0RXid0087rx+09999 +000099987')
2026-08-19 14:41:39,658 - pylabrobot.hamilton.star.driver.features.x_arm - DEBUG - the left X-arm was sent to 1000.0 mm and came to rest at 999.9 mm
2026-08-19 14:41:39,661 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid0088'
2026-08-19 14:41:39,670 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0RXid0088rx+09999 +000099992')
2026-08-19 14:41:39,673 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0XPid0089la09500lr3lw7'
2026-08-19 14:41:40,635 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0XPid0089er00')
2026-08-19 14:41:40,639 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid0090'
2026-08-19 14:41:40,649 

  1000.0      -0.10      -0.30


## 12- The autoload's X resolution

The driver reads the scanner X drive as 0.1 mm per increment. This machine has no `I0 QU` to ask
(`er30`, unknown command) and `I0 RA raau` answers `au0`, which the mapping calls 0.1 mm/step - so
the value is inferred, not measured.

Tracks are 22.5 mm apart, so moving between two tracks and reading the position each time measures
it directly: a unit that is really 0.125 mm/step would report the same travel 25% wide.

**This moves the autoload.** It travels along the front of the deck; nothing else moves.


In [19]:
if star.autoload is None:
  print("no autoload on this machine")
else:
  first, second = 10, 20
  await star.autoload.move_to_track(first)
  x_first = await star.autoload.request_x_position()
  await star.autoload.move_to_track(second)
  x_second = await star.autoload.request_x_position()

  tracks = second - first
  measured = (x_second - x_first) / tracks
  print(f"track {first} at {x_first:.2f} mm, track {second} at {x_second:.2f} mm")
  print(f"measured {measured:.3f} mm per track, against 22.5 mm expected")
  print(f"so the drive resolution is {0.1 * 22.5 / measured:.4f} mm per increment, read as 0.1")

  await star.autoload.park()

2026-08-19 14:41:41,566 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'I0RZid0093'
2026-08-19 14:41:41,581 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'I0RZid0093rz+0000 +0000')
2026-08-19 14:41:41,587 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'I0XPid0094xp10xv2500xr3xw7'
2026-08-19 14:41:45,950 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'I0XPid0094er00')
2026-08-19 14:41:45,955 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'I0RXid0095'
2026-08-19 14:41:45,965 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'I0RXid0095rx+02025 +02025')
2026-08-19 14:41:45,970 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'I0RZid0096'
2026-08-19 14:41:45,979 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'I0RZid0096rz+0000 +0000')
2026-08-19 14:41:45,985 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'I0XPid0097xp20xv2500xr3xw7'
2026-08-19 14:41:47,294 - pylabrobot.io.usb - IO - [0x8af:0x8000

track 10 at 202.50 mm, track 20 at 427.50 mm
measured 22.500 mm per track, against 22.5 mm expected
so the drive resolution is 0.1000 mm per increment, read as 0.1


2026-08-19 14:41:50,783 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'I0XPid0100er00')


## 13- Can we close the loop ourselves?

`X0 XP` lands within 0.3 mm and `C0 JX` is consistently 0.3 mm short, so neither command closes a
position loop. The question is whether we can: does re-commanding move the arm at all, and does
correcting by the measured error converge?

Two passes per target. The first re-sends the same target and watches whether anything changes; the
second commands `target + error`, which should walk the residual out if the drive responds to small
corrections at all. Note the read itself jitters by one increment, so 0.1 mm is the noise floor and
a tolerance below that will never be met.

**This moves the arm.** Gated on `allow_x_arm_move`.


In [20]:
tolerance = 0.15  # mm, above the read's own one-increment jitter
max_attempts = 4

if not allow_x_arm_move:
  print("skipped. set allow_x_arm_move = True to run")
else:
  for target in (500.0, 800.0):
    await star.x_arm.move_x(target - 50.0)  # approach from the same side every time
    await star.x_arm.move_x(target)
    print(f"\ntarget {target} mm")

    print("  re-sending the same target:")
    for attempt in range(max_attempts):
      reached = await star.x_arm.request_position()
      print(f"    attempt {attempt}: at {reached:.1f} mm, error {reached - target:+.1f} mm")
      if abs(reached - target) <= tolerance:
        break
      await star.x_arm.move_x(target)

    await star.x_arm.move_x(target - 50.0)
    await star.x_arm.move_x(target)
    print("  correcting by the measured error:")
    for attempt in range(max_attempts):
      reached = await star.x_arm.request_position()
      error = target - reached
      print(f"    attempt {attempt}: at {reached:.1f} mm, error {-error:+.1f} mm")
      if abs(error) <= tolerance:
        break
      await star.x_arm.move_x(target + error)

2026-08-19 14:41:50,797 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0XPid0101la04500lr3lw7'
2026-08-19 14:41:52,805 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0XPid0101er00')
2026-08-19 14:41:52,811 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid0102'
2026-08-19 14:41:52,820 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0RXid0102rx+04501 +000045013')
2026-08-19 14:41:52,824 - pylabrobot.hamilton.star.driver.features.x_arm - DEBUG - the left X-arm was sent to 450.0 mm and came to rest at 450.1 mm
2026-08-19 14:41:52,827 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0XPid0103la05000lr3lw7'
2026-08-19 14:41:53,740 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0XPid0103er00')
2026-08-19 14:41:53,745 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid0104'
2026-08-19 14:41:53,755 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0RXid0104rx+04998 +000049978')
2026-08-19 1


target 500.0 mm
  re-sending the same target:
    attempt 0: at 499.8 mm, error -0.2 mm


2026-08-19 14:41:53,985 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0XPid0106er00')
2026-08-19 14:41:53,990 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid0107'
2026-08-19 14:41:53,999 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0RXid0107rx+04999 +000049993')
2026-08-19 14:41:54,002 - pylabrobot.hamilton.star.driver.features.x_arm - DEBUG - the left X-arm was sent to 500.0 mm and came to rest at 499.9 mm
2026-08-19 14:41:54,005 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid0108'
2026-08-19 14:41:54,014 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0RXid0108rx+05000 +000049996')
2026-08-19 14:41:54,019 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0XPid0109la04500lr3lw7'


    attempt 1: at 500.0 mm, error +0.0 mm


2026-08-19 14:41:54,981 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0XPid0109er00')
2026-08-19 14:41:54,987 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid0110'
2026-08-19 14:41:54,996 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0RXid0110rx+04502 +000045021')
2026-08-19 14:41:55,000 - pylabrobot.hamilton.star.driver.features.x_arm - DEBUG - the left X-arm was sent to 450.0 mm and came to rest at 450.2 mm
2026-08-19 14:41:55,003 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0XPid0111la05000lr3lw7'
2026-08-19 14:41:55,966 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0XPid0111er00')
2026-08-19 14:41:55,971 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid0112'
2026-08-19 14:41:55,981 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0RXid0112rx+04998 +000049980')
2026-08-19 14:41:55,985 - pylabrobot.hamilton.star.driver.features.x_arm - DEBUG - the left X-arm was sent to 500.0 

...

2026-08-19 14:42:01,365 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0XPid0130er00')
2026-08-19 14:42:01,370 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid0131'
2026-08-19 14:42:01,379 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0RXid0131rx+07996 +000079959')
2026-08-19 14:42:01,383 - pylabrobot.hamilton.star.driver.features.x_arm - DEBUG - the left X-arm was sent to 799.7 mm and came to rest at 799.6 mm
2026-08-19 14:42:01,386 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid0132'
2026-08-19 14:42:01,396 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0RXid0132rx+07996 +000079959')
2026-08-19 14:42:01,400 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0XPid0133la08004lr3lw7'


    attempt 2: at 799.6 mm, error -0.4 mm


2026-08-19 14:42:01,662 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0XPid0133er00')
2026-08-19 14:42:01,667 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid0134'
2026-08-19 14:42:01,677 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0RXid0134rx+08006 +000080058')
2026-08-19 14:42:01,680 - pylabrobot.hamilton.star.driver.features.x_arm - DEBUG - the left X-arm was sent to 800.4 mm and came to rest at 800.6 mm
2026-08-19 14:42:01,683 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid0135'
2026-08-19 14:42:01,692 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0RXid0135rx+08006 +000080057')
2026-08-19 14:42:01,696 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0XPid0136la07994lr3lw7'


    attempt 3: at 800.6 mm, error +0.6 mm


2026-08-19 14:42:02,010 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0XPid0136er00')
2026-08-19 14:42:02,014 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid0137'
2026-08-19 14:42:02,024 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0RXid0137rx+07993 +000079925')
2026-08-19 14:42:02,027 - pylabrobot.hamilton.star.driver.features.x_arm - DEBUG - the left X-arm was sent to 799.4 mm and came to rest at 799.3 mm


## 13b- Why is each X move short: undershoot, or an offset?

`X0 XP` lands 0.2 mm short of a long move and exactly on a short one, so re-sending the target
converges. `C0 JX` was short by 0.30 mm at all three targets, which is too consistent for
deceleration undershoot - but it was only ever approached from below, so the two explanations were
never separated.

They differ in how the error is signed.

| | error follows | approached from below | approached from above | from rest |
|---|---|---|---|---|
| undershoot | the direction of travel | lands low | lands high | no error |
| offset | the axis, always the same way | lands low | lands low | lands low |

So the experiment is the same target reached from both sides, and from rest, through each command.
Distance is varied too, since undershoot grows with speed and a short move showed none.

**This moves the arm.** Gated on `allow_x_arm_move`. Every start position is settled with `X0 XP`,
re-sent until it is within tolerance, so each measurement begins from a known place.


In [21]:
target = 600.0
distances = (0.5, 100.0)
tolerance = 0.15

if not allow_x_arm_move:
  print("skipped. set allow_x_arm_move = True to run")
else:

  async def at() -> float:
    return await star.x_arm.request_position()

  async def settle_at(x: float) -> float:
    """Put the arm on x with the command that converges, and say where it ended up."""
    for _ in range(4):
      await star.x_arm.move_x(x)
      if abs(await at() - x) <= tolerance:
        break
    return await at()

  async def send(command: str, x: float) -> None:
    if command == "X0 XP":
      await star.x_arm.move_x(x)
    else:
      await star.driver.send_command(module="C0", command="JX", xs=f"{round(x * 10):05}")

  print(f"{'command':8s} {'from':>8} {'distance':>9} {'started':>9} {'reached':>9} {'error':>7}")
  for command in ("X0 XP", "C0 JX"):
    for distance in distances:
      for direction, start in (("below", target - distance), ("above", target + distance)):
        started = await settle_at(start)
        await send(command, target)
        reached = await at()
        print(
          f"{command:8s} {direction:>8} {distance:9.1f} {started:9.1f} {reached:9.1f}"
          f" {reached - target:+7.1f}"
        )
    # from rest on the target itself: undershoot has nothing to undershoot
    started = await settle_at(target)
    await send(command, target)
    reached = await at()
    print(
      f"{command:8s} {'rest':>8} {0.0:9.1f} {started:9.1f} {reached:9.1f} {reached - target:+7.1f}"
    )

  print("\nre-sending the same target, through each command:")
  for command in ("X0 XP", "C0 JX"):
    await settle_at(target - 100.0)
    print(f"  {command}")
    for attempt in range(4):
      await send(command, target)
      reached = await at()
      print(f"    attempt {attempt}: at {reached:9.1f} mm, error {reached - target:+.1f} mm")
      if abs(reached - target) <= tolerance:
        break

command      from  distance   started   reached   error


2026-08-19 14:42:02,065 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0XPid0138la05995lr3lw7'
2026-08-19 14:42:03,626 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0XPid0138er00')
2026-08-19 14:42:03,631 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid0139'
2026-08-19 14:42:03,641 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0RXid0139rx+05996 +000059960')
2026-08-19 14:42:03,644 - pylabrobot.hamilton.star.driver.features.x_arm - DEBUG - the left X-arm was sent to 599.5 mm and came to rest at 599.6 mm
2026-08-19 14:42:03,646 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid0140'
2026-08-19 14:42:03,655 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0RXid0140rx+05996 +000059958')
2026-08-19 14:42:03,659 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid0141'
2026-08-19 14:42:03,669 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0RXid0141rx+05995 +000059954')
2026-08-19

X0 XP       below       0.5     599.5     600.0    +0.0


2026-08-19 14:42:04,231 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0XPid0145er00')
2026-08-19 14:42:04,236 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid0146'
2026-08-19 14:42:04,245 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0RXid0146rx+06006 +000060056')
2026-08-19 14:42:04,248 - pylabrobot.hamilton.star.driver.features.x_arm - DEBUG - the left X-arm was sent to 600.5 mm and came to rest at 600.6 mm
2026-08-19 14:42:04,251 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid0147'
2026-08-19 14:42:04,260 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0RXid0147rx+06006 +000060056')
2026-08-19 14:42:04,264 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid0148'
2026-08-19 14:42:04,274 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0RXid0148rx+06006 +000060056')
2026-08-19 14:42:04,278 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0XPid0149la06000lr3lw7'
2026-08-19

X0 XP       above       0.5     600.6     599.9    -0.1


...

    attempt 1: at     600.0 mm, error +0.0 mm


2026-08-19 14:42:22,487 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0XPid0225er00')
2026-08-19 14:42:22,492 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid0226'
2026-08-19 14:42:22,502 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0RXid0226rx+05002 +000050020')
2026-08-19 14:42:22,505 - pylabrobot.hamilton.star.driver.features.x_arm - DEBUG - the left X-arm was sent to 500.0 mm and came to rest at 500.2 mm
2026-08-19 14:42:22,508 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid0227'
2026-08-19 14:42:22,517 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0RXid0227rx+05002 +000050015')
2026-08-19 14:42:22,521 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0XPid0228la05000lr3lw7'
2026-08-19 14:42:22,732 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0XPid0228er00')
2026-08-19 14:42:22,736 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid0229'
2026-08-19 14:42:22,746 -

  C0 JX


2026-08-19 14:42:24,357 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'C0JXid0232er00/00')
2026-08-19 14:42:24,362 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid0233'
2026-08-19 14:42:24,372 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0RXid0233rx+06000 +000059995')


    attempt 0: at     600.0 mm, error +0.0 mm


## 13c- Why did the same command measure differently?

Section 11 found `C0 JX` short by 0.30 mm at 500, 800 and 1000 mm. Section 13b found it accurate at
600 mm from either side, at both 0.5 and 100 mm, and from rest. Both read the position 3-6 ms after
a reply that only comes once the move is over, so neither was reading a moving arm.

Two differences remain between the protocols, and this separates them.

- **Section 11 moved exactly 50 mm every time.** 13b used 0.5 mm and 100 mm.
- **Section 11 started where `X0 XP` had just left the arm**, which is itself 0.2 mm off target,
  rather than from a settled position.

So each target is reached by `JX` twice: once from a start `XP` left behind, exactly as section 11
did, and once from the same start settled to the millimetre. If only the first is short, what
matters is the state `XP` leaves; if both are, it is the 50 mm distance.

**This moves the arm.** Gated on `allow_x_arm_move`.


In [22]:
if not allow_x_arm_move:
  print("skipped. set allow_x_arm_move = True to run")
else:

  async def at() -> float:
    return await star.x_arm.request_position()

  async def settle_at(x: float) -> float:
    for _ in range(4):
      await star.x_arm.move_x(x)
      if abs(await at() - x) <= 0.15:
        break
    return await at()

  async def jump_to(x: float) -> None:
    await star.driver.send_command(module="C0", command="JX", xs=f"{round(x * 10):05}")

  print(f"{'target':>8} {'start':>22} {'started':>9} {'reached':>9} {'error':>7}")
  for target in (500.0, 800.0, 1_000.0):
    # exactly as section 11: one XP move to 50 mm below, wherever that lands
    await star.x_arm.move_x(target)
    await star.x_arm.move_x(target - 50.0)
    started = await at()
    await jump_to(target)
    reached = await at()
    print(
      f"{target:8.1f} {'as XP left it':>22} {started:9.1f} {reached:9.1f} {reached - target:+7.1f}"
    )

    # the same start, settled first
    started = await settle_at(target - 50.0)
    await jump_to(target)
    reached = await at()
    print(f"{target:8.1f} {'settled':>22} {started:9.1f} {reached:9.1f} {reached - target:+7.1f}")

2026-08-19 14:42:24,396 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0XPid0234la05000lr3lw7'


  target                  start   started   reached   error


2026-08-19 14:42:25,605 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0XPid0234er00')
2026-08-19 14:42:25,610 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid0235'
2026-08-19 14:42:25,619 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0RXid0235rx+05002 +000050020')
2026-08-19 14:42:25,623 - pylabrobot.hamilton.star.driver.features.x_arm - DEBUG - the left X-arm was sent to 500.0 mm and came to rest at 500.2 mm
2026-08-19 14:42:25,626 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0XPid0236la04500lr3lw7'
2026-08-19 14:42:26,541 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0XPid0236er00')
2026-08-19 14:42:26,545 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid0237'
2026-08-19 14:42:26,555 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0RXid0237rx+04502 +000045020')
2026-08-19 14:42:26,558 - pylabrobot.hamilton.star.driver.features.x_arm - DEBUG - the left X-arm was sent to 450.0 

   500.0          as XP left it     450.2     499.7    -0.3


2026-08-19 14:42:28,384 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0XPid0241er00')
2026-08-19 14:42:28,389 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid0242'
2026-08-19 14:42:28,398 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0RXid0242rx+04502 +000045022')
2026-08-19 14:42:28,402 - pylabrobot.hamilton.star.driver.features.x_arm - DEBUG - the left X-arm was sent to 450.0 mm and came to rest at 450.2 mm
2026-08-19 14:42:28,405 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid0243'
2026-08-19 14:42:28,413 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0RXid0243rx+04502 +000045022')
2026-08-19 14:42:28,417 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0XPid0244la04500lr3lw7'
2026-08-19 14:42:28,635 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0XPid0244er00')
2026-08-19 14:42:28,640 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid0245'
2026-08-19 14:42:28,649 -

...

   800.0                settled     750.1     800.0    +0.0


2026-08-19 14:42:37,425 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0XPid0266er00')
2026-08-19 14:42:37,430 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid0267'
2026-08-19 14:42:37,440 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0RXid0267rx+09999 +000099989')
2026-08-19 14:42:37,443 - pylabrobot.hamilton.star.driver.features.x_arm - DEBUG - the left X-arm was sent to 1000.0 mm and came to rest at 999.9 mm
2026-08-19 14:42:37,446 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0XPid0268la09500lr3lw7'
2026-08-19 14:42:38,357 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0XPid0268er00')
2026-08-19 14:42:38,362 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid0269'
2026-08-19 14:42:38,371 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0RXid0269rx+09502 +000095017')
2026-08-19 14:42:38,374 - pylabrobot.hamilton.star.driver.features.x_arm - DEBUG - the left X-arm was sent to 950.0

  1000.0          as XP left it     950.2     999.7    -0.3


2026-08-19 14:42:40,196 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0XPid0273er00')
2026-08-19 14:42:40,202 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid0274'
2026-08-19 14:42:40,211 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0RXid0274rx+09502 +000095023')
2026-08-19 14:42:40,214 - pylabrobot.hamilton.star.driver.features.x_arm - DEBUG - the left X-arm was sent to 950.0 mm and came to rest at 950.2 mm
2026-08-19 14:42:40,216 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid0275'
2026-08-19 14:42:40,226 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0RXid0275rx+09502 +000095023')
2026-08-19 14:42:40,229 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0XPid0276la09500lr3lw7'
2026-08-19 14:42:40,441 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0XPid0276er00')
2026-08-19 14:42:40,445 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid0277'
2026-08-19 14:42:40,455 -

  1000.0                settled     950.0    1000.0    +0.0


## 14- The instrument configuration, and what writing it would mean

`C0 AK` writes the machine's non-volatile configuration. It takes **21 parameters**, each with a
default, and the master's convention is that an unsent parameter takes its default - so a partial
`AK` does not change one field, it rewrites all of them. Sending `kb` alone would declare no
channels, no 96-head, a different arm width and a different waste position.

Every one of the 21 is readable: `C0 RM` answers `kb` and `kp`, `C0 QM` the other 19. The cell below
reads them, rebuilds the command that would restore exactly what the machine says today, and shows
what changes if the front cover monitoring bit is set. **It sends nothing.**

Why we would want to: with `kb` bit 2 clear, `C0 QC` answered `qc1` with the cover open, so the
master is not reading the switch. Setting the bit is the only way to find out whether `QC` reports
the cover on a machine that declares the monitoring - and it is also what makes the machine abort a
run when the cover opens, which is why it was turned off in the first place.


In [23]:
import re

# The 21 parameters AK takes, in the order the specification lists them.
AK_PARAMETERS = "ka ke xt xa xw kb xl xn xr xo xm xx xu xv kp ys kl km ym yu yx".split()


def read_fields(reply: str) -> dict:
  """The two-letter fields in a reply, as the machine wrote them."""
  return dict(re.findall(r"([a-z]{2})([0-9A-Fa-f]+)", reply.split("er00/00", 1)[-1]))


if protocol_mode != "execution":
  raise SystemExit("nothing to read: a simulated machine has no configuration to rebuild")

machine = await star.driver.send_command(module="C0", command="RM")
extended = await star.driver.send_command(module="C0", command="QM")
read = {**read_fields(extended), **read_fields(machine)}

missing = [name for name in AK_PARAMETERS if name not in read]
print(f"read {len(AK_PARAMETERS) - len(missing)} of {len(AK_PARAMETERS)} parameters")
if missing:
  print(f"MISSING, so a safe write is not possible: {missing}")
else:
  as_it_stands = "".join(f"{name}{read[name]}" for name in AK_PARAMETERS)
  print(f"\nrestores exactly what the machine says now:\n  C0AK{as_it_stands}")

  with_monitoring = dict(read)
  with_monitoring["kb"] = f"{int(read['kb'], 16) | 0b100:02X}"
  proposed = "".join(f"{name}{with_monitoring[name]}" for name in AK_PARAMETERS)
  print(f"\nwith the front cover monitoring bit set:\n  C0AK{proposed}")
  print(f"\nkb {read['kb']} -> {with_monitoring['kb']}")
  print(
    "everything else identical:",
    as_it_stands.replace(f"kb{read['kb']}", "")
    == proposed.replace(f"kb{with_monitoring['kb']}", ""),
  )

2026-08-19 14:42:41,716 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'C0RMid0282'
2026-08-19 14:42:41,776 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'C0RMid0282er00/00kb0Bkp08 C00000 X00000 P10000 P20000 P30000 P40000 P50000 P60000 P70000 P80000 I00000 R00000 H00000')
2026-08-19 14:42:41,780 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'C0QMid0283'
2026-08-19 14:42:41,819 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'C0QMid0283er00/00ka010003xt54xa54xw13400xl07xr00xm03500xx11400ys090xu3540xv3700yu0060kl360kc0yx0060ke00000000xn00xo00ym6065kr0km360')


read 21 of 21 parameters

restores exactly what the machine says now:
  C0AKka010003ke00000000xt54xa54xw13400kb0Bxl07xn00xr00xo00xm03500xx11400xu3540xv3700kp08ys090kl360km360ym6065yu0060yx0060

with the front cover monitoring bit set:
  C0AKka010003ke00000000xt54xa54xw13400kb0Fxl07xn00xr00xo00xm03500xx11400xu3540xv3700kp08ys090kl360km360ym6065yu0060yx0060

kb 0B -> 0F
everything else identical: True


### Writing it

Only with `allow_configuration_write = True`, set in the cell itself so it cannot be reached by
running the notebook top to bottom. It writes the full command built above, reads the configuration
back, and prints the restore command in case the read-back does not match.

Keep the restore line from the cell above. If anything goes wrong, sending it puts the machine back.


In [24]:
allow_configuration_write = False

if not allow_configuration_write:
  print("skipped. this rewrites the machine's non-volatile configuration")
elif missing:
  print("refused: not every parameter could be read")
else:
  print(f"restore command, keep this:\n  C0AK{as_it_stands}\n")
  print(await star.driver.send_raw_command(f"C0AK{proposed}"))

  after = {
    **read_fields(await star.driver.send_command(module="C0", command="QM")),
    **read_fields(await star.driver.send_command(module="C0", command="RM")),
  }
  for name in AK_PARAMETERS:
    if after.get(name) != with_monitoring.get(name):
      print(f"  {name}: wrote {with_monitoring.get(name)}, reads back {after.get(name)}")
  print(
    "read back identical to what was written:",
    all(after.get(n) == with_monitoring.get(n) for n in AK_PARAMETERS),
  )

skipped. this rewrites the machine's non-volatile configuration


## 15- Raw command escape hatch

Anything not yet wrapped in a named method can be sent directly. **Only send commands you have
confirmed are read-only** - this bypasses every guard in the driver.

In [25]:
# C0 RF - request the master's firmware version. Read-only.
print(await star.driver.send_command(module="C0", command="RF"))

# the same thing as a raw string, id included
# print(await star.driver.send_raw_command("C0RFid9999"))

# Read the autoload's stored configuration, whose first field is the scanner X-drive resolution:
# 0 => 0.1 mm/step, 1 => 0.125 mm/step on a pilot-lot unit. The driver hardcodes 0.1, which is
# only right for the units that have it.
#
# Two candidates, because the read differs by autoload generation: `QU` on the later firmware,
# and the generic parameter read `RA` on the generation this machine reports. Both are
# read-only. Whichever answers, its reply is the shape a
# request_x_resolution() would parse - so print it raw.
for attempt in ("I0QUid9990", "I0RAid9991raau"):
  try:
    print(attempt, "->", await star.driver.send_raw_command(attempt))
  except Exception as e:  # noqa: BLE001 - whatever the machine says is the answer
    print(attempt, "-> refused:", e)

2026-08-19 14:42:41,864 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'C0RFid0284'
2026-08-19 14:42:41,877 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'C0RFid0284er00/00rf7.6S 25 2021_11_05 (GRU C0)')
2026-08-19 14:42:41,881 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'I0QUid9990'


C0RFid0284er00/00rf7.6S 25 2021_11_05 (GRU C0)


2026-08-19 14:42:41,890 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'I0QUid9990er30')
2026-08-19 14:42:41,893 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'I0RAid9991raau'
2026-08-19 14:42:41,905 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'I0RAid9991au0 0 0 0 0')


I0QUid9990 -> refused: {'Auto Load': UnknownHamiltonError('Unknown command')}, I0QUid9990er30
I0RAid9991raau -> I0RAid9991au0 0 0 0 0


## 16- What is ported, and what is not

Every module hangs off the same reply router, so each remaining one is a module to add rather
than new plumbing.

| Module | Node | State |
|---|---|---|
| Master | `C0` | configuration, initialization, tip presence |
| Pipetting channels | `P1`-`PG` | firmware, width, installed hardware, initialization |
| X-drives | `X0` | firmware, absolute move |
| 96-head | `H0` | firmware, hardware, drive parameters, retract, initialization |
| iSWAP | `R0` | not ported |
| Autoload | `I0` | not ported |
| Wash stations, pumps | `W1`/`W2`, `HW`/`HU`/`HV` | not ported |

On a machine with a 96-head, setup retracts it to Z safety and probes how far it reaches. If the
head reports itself uninitialized, setup says so rather than guessing: initializing it ejects
whatever is mounted, so it needs the position to eject at - `head96.initialize(x, y, z)`.

## 17- Teardown

In [26]:
await star.stop()
print("disconnected. connected:", star.driver.connected, "| setup done:", star.driver.setup_done)

# The log is append-only and stays open for the rest of the session - nothing to close.

2026-08-19 14:42:41,918 - pylabrobot.io.usb - WARNING - Closing connection to USB device.


disconnected. connected: False | setup done: False
